# LIBRERIAS

In [1]:
# ----------------- Imports y configuración básica -----------------
import os
import sagemaker
from sagemaker.workflow.function_step import step  
from sagemaker.workflow.pipeline import Pipeline 
from sagemaker.workflow.condition_step import ConditionStep 
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo 
from sagemaker.workflow.fail_step import FailStep 
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import LabelEncoder 
from sklearn.ensemble import RandomForestClassifier 
from imblearn.over_sampling import SMOTE 
from sklearn.metrics import f1_score  
import pandas as pd  
import joblib 
from sagemaker import get_execution_role  

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


# INFORMACION DEL ENTORNO

In [2]:
role = get_execution_role()  # Obtener el rol de ejecución de SageMaker
user = "mlops"  # Nombre del usuario
default_bucket = "myawsbucket.3.2025"  # Nombre del bucket de S3
default_prefix = f"sagemaker/attrition-detection/{user}"  # Prefijo para la ruta en S3
sagemaker_session = sagemaker.Session(default_bucket=default_bucket, default_bucket_prefix=default_prefix)  # Iniciar la sesión de SageMaker
instance_type = "ml.m5.large"  # Tipo de instancia para los pasos del pipeline
pipeline_name = "pipeline-attrition-modelado"  # Nombre del pipeline
model_name = f"attrition-detection-{user}"  # Nombre del modelo
train_s3_path = f"s3://{default_bucket}/sagemaker/attrition-detection/mlops-utec-data/train_clean.csv"  # Ruta del archivo de entrenamiento en S3
current_dir = os.getcwd()  # Obtener el directorio actual
requirements_path = os.path.join(current_dir, "model_training_requirements.txt")  # Ruta al archivo de requerimientos

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


# MODEL TRAINING 

In [3]:
@step(name="ModelTrainingStep", instance_type=instance_type, dependencies=requirements_path)
def model_training(experiment_name: str) -> tuple[str, str, pd.DataFrame, pd.Series]:
    """
    Paso para cargar los datos, preprocesarlos, entrenar el modelo y guardar el modelo entrenado en S3.
    """
    TARGET_COL = "ATTRITION"  # Columna objetivo
    SEED = 42  # Semilla para la aleatoriedad
    TRAIN_SPLIT = 0.7  # Porcentaje de datos para entrenamiento

    # Cargar los datos de entrenamiento
    df = pd.read_csv(train_s3_path)
    df.drop(columns=['ID_CORRELATIVO', 'CODMES_x', 'CODMES_y'], inplace=True)  # Eliminar columnas innecesarias

    # Codificar las columnas categóricas
    le = LabelEncoder()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = le.fit_transform(df[col].astype(str))

    # Separar características (X) y etiqueta (y)
    X = df.drop(columns=TARGET_COL)
    y = df[TARGET_COL]

    # Dividir los datos en entrenamiento y prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=TRAIN_SPLIT, random_state=SEED, stratify=y)
    
    # Balancear las clases con SMOTE
    smote = SMOTE(random_state=SEED)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    # Subir los datos de prueba a S3
    test_data_s3_path = f"s3://{default_bucket}/{default_prefix}/test_data/test_data.csv"
    pd.concat([X_test, y_test], axis=1).to_csv(test_data_s3_path, index=False)  # Guardar los datos de prueba en S3

    return test_data_s3_path, experiment_name, X_train_smote, y_train_smote

# MODEL EVALUATION

In [4]:
@step(name="ModelEvaluationStep", instance_type=instance_type, dependencies=requirements_path)
def evaluate(test_data_s3_path: str, experiment_name: str, X_train: pd.DataFrame, y_train: pd.Series) -> dict:
    """
    Paso para entrenar el modelo y evaluar su rendimiento utilizando F1 score.
    """
    SEED = 42
    model = RandomForestClassifier(random_state=SEED, n_estimators=200)  # Definir el modelo
    model.fit(X_train, y_train)  # Entrenar el modelo

    # Leer los datos de prueba
    df_test = pd.read_csv(test_data_s3_path)
    X_test = df_test.drop(columns="ATTRITION")
    y_true = df_test["ATTRITION"]

    # Realizar las predicciones
    y_pred = model.predict(X_test)

    # Calcular el F1 score
    f1 = f1_score(y_true, y_pred)
    return {"f1_score": f1}

# REGISTRO DEL MODELO 

In [5]:
@step(name="ModelRegistrationStep", instance_type=instance_type, dependencies=requirements_path)
def register(model_artifact_path: str):
    """
    Paso para registrar el modelo entrenado en SageMaker.
    """
    print(f"El modelo ha sido registrado exitosamente desde {model_artifact_path}")

# PIPELINE

In [6]:
# Nombre del experimento
experiment_name = "modelado"

# Crear los pasos
model_training_step = model_training(experiment_name=experiment_name)

evaluate_step = evaluate(
    test_data_s3_path=model_training_step[0],
    experiment_name=model_training_step[1],
    X_train=model_training_step[2],
    y_train=model_training_step[3]
)

register_step = register(model_artifact_path=model_training_step[2])

# Paso condicional para registrar el modelo si el F1 score es suficiente
conditional_register_step = ConditionStep(
    name="ConditionalRegisterStep",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=evaluate_step["f1_score"],  # Verificar que el F1 score es mayor o igual a 0.2
            right=0.2
        )
    ],
    if_steps=[register_step],  # Si es suficiente, registrar el modelo
    else_steps=[FailStep(name="FailIfLowF1", error_message="Model performance is not good enough")]  # Si no, fallar
)

# Ensamblar el pipeline
pipeline = Pipeline(
    name=pipeline_name,  # Nombre del pipeline
    steps=[
        model_training_step,  # Paso de entrenamiento
        evaluate_step,  # Paso de evaluación
        conditional_register_step  # Paso condicional de registro
    ]
)

# Crear o actualizar el pipeline
pipeline.upsert(role_arn=role)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:40                                                                                   │
│                                                                                                  │
│   37 )                                                                                           │
│   38                                                                                             │
│   39 # Crear o actualizar el pipeline                                                            │
│ ❱ 40 pipeline.upsert(role_arn=role)                                                              │
│   41                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:297 in upsert             │
│                                                                                                  │
│    294 │   │   │   error_code = ce.response["Error"]["Code"]                                     │
│    295 │   │   │   error_message = ce.response["Error"]["Message"]                               │
│    296 │   │   │   if not (error_code == "ValidationException" and "already exists" in error_me  │
│ ❱  297 │   │   │   │   raise ce                                                                  │
│    298 │   │   │   # already exists                                                              │
│    299 │   │   │   response = self.update(role_arn, description, parallelism_config=parallelism  │
│    300 │   │   │   # add new tags to existing resource                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:292 in upsert             │
│                                                                                                  │
│    289 │   │   │   # after fetching the config.                                                  │
│    290 │   │   │   raise ValueError("An AWS IAM role is required to create or update a Pipeline  │
│    291 │   │   try:                                                                              │
│ ❱  292 │   │   │   response = self.create(role_arn, description, tags, parallelism_config)       │
│    293 │   │   except ClientError as ce:                                                         │
│    294 │   │   │   error_code = ce.response["Error"]["Code"]                                     │
│    295 │   │   │   error_message = ce.response["Error"]["Message"]                               │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:164 in create             │
│                                                                                                  │
│    161 │   │   tags = format_tags(tags)                                                          │
│    162 │   │   tags = _append_project_tags(tags)                                                 │
│    163 │   │   tags = self.sagemaker_session._append_sagemaker_config_tags(tags, PIPELINE_TAGS_  │
│ ❱  164 │   │   kwargs = self._create_args(role_arn, description, parallelism_config)             │
│    165 │   │   update_args(                                                                      │
│    166 │   │   │   kwargs,                                                                       │
│    167 │   │   │   Tags=tags,                                                                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/

In [7]:
import boto3
from botocore.exceptions import ClientError

# Crea un cliente de STS para asumir el rol
sts_client = boto3.client('sts')

# Definir el ARN del rol y el nombre de la sesión
role_arn = 'arn:aws:iam::880138931512:role/service-role/AmazonSageMaker-ExecutionRole-20250618T220865'
session_name = 'SageMakerSession'

# Función para obtener un token temporal
def get_temp_credentials(role_arn, session_name):
    try:
        # Asumir el rol y obtener credenciales temporales
        response = sts_client.assume_role(
            RoleArn=role_arn,
            RoleSessionName=session_name
        )

        # Extraer las credenciales de la respuesta
        credentials = response['Credentials']

        print("Nuevo token temporal obtenido correctamente:")
        print(f"AccessKeyId: {credentials['AccessKeyId']}")
        print(f"SecretAccessKey: {credentials['SecretAccessKey']}")
        print(f"SessionToken: {credentials['SessionToken']}")

        # Configurar boto3 con las nuevas credenciales temporales
        boto3.setup_default_session(
            aws_access_key_id=credentials['AccessKeyId'],
            aws_secret_access_key=credentials['SecretAccessKey'],
            aws_session_token=credentials['SessionToken']
        )

        print("Credenciales configuradas correctamente.")
        
    except ClientError as e:
        print(f"Error al obtener el token temporal: {e}")

# Ejecutar la función
get_temp_credentials(role_arn, session_name)

# Verificar si las credenciales funcionan
try:
    # Crear un cliente de SageMaker
    sagemaker_client = boto3.client('sagemaker')
    
    # Listar las instancias de notebooks de SageMaker como prueba
    response = sagemaker_client.list_notebook_instances()
    print("Instancias de notebook de SageMaker:")
    print(response)
    
except ClientError as e:
    print(f"Error al acceder a SageMaker: {e}")


INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Error al obtener el token temporal: An error occurred (InvalidClientTokenId) when calling the AssumeRole operation: The security token included in the request is invalid.
Error al acceder a SageMaker: An error occurred (UnrecognizedClientException) when calling the ListNotebookInstances operation: The security token included in the request is invalid.
